In [1]:
#!pip install sentence_transformers
#!pip install ag2[retrievechat]
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

# Setting base for Retrive Chat

In [2]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

C:\Users\manju\anaconda3\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.10.20). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\manju\anaconda3\lib\site-packages\autogen\oai\gemini.py:65: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Patching name='PARAMS_MAPPING', member={'max_tokens': 'max_output_tokens', 'stop_sequences': 'stop_sequences', 'temperature': 'temperature', 'top_p': 'top_p', 'top_k': 'top_k', 'max_output_tokens': 'max_output_tokens'}, patched={'max_tokens': 'max_output_tokens', 'stop_sequences': 'stop_sequences', 'temperature': 'temperature', 'top_p': 'top_p', 'top_k': 'top_k', 'max_output_tokens': 'max_output_tokens'}
Patching name='__init__', member=<function GeminiClient.__init__ at 0x000001B3804DF370>, patched=<function function.__call__ at 0x000001B3804DFC70>
Patching name='_concat_parts', member=<function GeminiClient._concat_parts at 0x000001B3804DF6D0>, patched=<function function.__call__ at 0x000001B3804DFD00>
Patching name='_convert_json_response', member=<function GeminiClient._convert_json_response at 0x000001B3804DF7F0>, patched=<function function.__call__ at 0x000001B3804DFD90>
Patching name='_create_gemini_function_declaration', member=<function GeminiClient._create_gemini_function_dec

C:\Users\manju\anaconda3\lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [3]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [4]:
config_list = [
    {
        # Let's choose the Meta's Llama 3.1 model (model names must match Ollama exactly)
        "model": "llama3.1",
        # We specify the API Type as 'ollama' so it uses the Ollama client class
        "api_type": "ollama",
        "stream": False,
        #"client_host": "http://192.168.0.1:11434",
    }
]

In [5]:
import agentops
agentops.init("954ecbcb-9555-470b-a2ad-a6999473648b")

🖇 AgentOps: [ConcurrentFuturesInstrumentor] Successfully instrumented concurrent.futures.ThreadPoolExecutor
🖇 AgentOps: You're on the agentops free plan 🤔
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\manju\anaconda3\lib\logging\__init__.py", line 1103, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\manju\anaconda3\lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f914' in position 66: character maps to <undefined>
Call stack:
  File "C:\Users\manju\anaconda3\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\manju\anaconda3\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\manju\anaconda3\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\manju\anaconda3\lib\sit

In [6]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [11]:
import pypdf

reader = pypdf.PdfReader(str("book.pdf"))

pages = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        pages.append(f"[Page {page_number}]\n{text}")

docs_content = "\n\n".join(pages)

print("Pages:", len(reader.pages))
print("Extracted characters:", len(docs_content))
print(docs_content[:1000])


Pages: 141
Extracted characters: 167098
[Page 2]
The 22
Immutable Laws
of Marketing
Violate Them at Y our Own Risk
Al Ries and Jack Trout
22 Laws of Marketing  10/31/02  12:23 PM  Page 2

[Page 3]
Dedicated to the elimination of
myths and misconceptions
from the marketing process
22 Laws of Marketing  10/31/02  12:23 PM  Page 3

[Page 4]
Contents
Introduction ix
1. The Law of Leadership 
2. The Law of the Category 
3. The Law of the Mind 
4. The Law of Perception 
5. The Law of Focus 
6. The Law of Exclusivity 
7. The Law of the Ladder 
8. The Law of Duality 
9. The Law of the Opposite 
10. The Law of Division 
11. The Law of Perspective 
12. The Law of Line Extension 
13. The Law of Sacrifice 
14. The Law of Attributes 
15. The Law of Candor 
16. The Law of Singularity 
17. The Law of Unpredictability 
18. The Law of Success 
19. The Law of Failure 
20.The Law of Hype 
21. The Law of Acceleration 
22. The Law of Resources 
Warning 
About the Author
Credits     
Copyright
About the Pub

In [13]:

from autogen import AssistantAgent

config_list = [
    {
        "model": "llama3.1:latest",
        "api_type": "openai",
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama"
    }
]

assistant = AssistantAgent(
    name="pdf_assistant",
    system_message=(
        "You answer questions using the supplied PDF text. "
        "Base answers on the document, cite page numbers when available, "
        "and say when the answer is not found in the document."
    ),
    llm_config={
        "config_list": config_list,
        "timeout": 600,
        "cache_seed": 42
    }
)

print("AutoGen PDF assistant created.")


AutoGen PDF assistant created.


Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [14]:

question = "What is this book about? Summarize its main points."

prompt = f"""
Answer the question using the PDF text below.

Question:
{question}

PDF text:
{docs_content[:20000]}

If the answer is not in the supplied text, say so.
Cite page numbers where possible.
"""

reply = assistant.generate_reply(
    messages=[{"role": "user", "content": prompt}]
)

print(reply)


[autogen.oai.client: 10-16 11:51:59] {521} WARNING - Model llama3.1:latest is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Here are the answers to the questions:

1. What is the name of the first brand of aspirin, the first brand of acetaminophen, and the first brand of ibuprofen?
The name of the first brand of aspirin, the first brand of acetaminophen, and the first brand of ibuprofen is "Bayer".

2. The Charles Lindbergh of the discount brokerage business is:
Charles Schwab

3. The second person to walk on the moon was:
Buzz Aldrin

4. The second person to run a four-minute mile was:
John Landy

5. The second president of the United States was:
John Adams

6. The second English muffin brand was:
Thomas' is not mentioned as the second English muffin brand, but rather it was the first brand, so the answer is unclear.

7. The second sports drink was:
Gatorad is not explicitly mentio

In [15]:
question = "What are the most important lessons in Chapter 2?"

prompt = f"""
Answer based on the following PDF text.
Question: {question}

PDF text:
{docs_content[:20000]}

Cite page numbers where possible. Do not invent information.
"""

print(assistant.generate_reply(
    messages=[{"role": "user", "content": prompt}]
))

[autogen.oai.client: 10-16 11:54:28] {521} WARNING - Model llama3.1:latest is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Here are the questions I can help you with:

1. What is the law that states the first brand into the market always leads, but other brands follow up to a point, as shown by the example of Advil, Nuprin, Medipren, and Motrin IB?
2. What is the name of the popular discount brokerage firm that bills itself as "America's largest discount broker"?
3. Who was the first person to walk on the moon, and who was second?
4. Who was the first person to run a four-minute mile, and who was second?
5. Who was the first president of the United States, and who was his successor?
6. What was the first brand of English muffin, and what was second?
7. What is the law that states if you can't be first in a category, set up a new category you can be first in, and provide an example 

Scenario 2: Answer a question based off docstrings w/o human feedback

In [16]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\manju\anaconda3\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\manju\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:

from autogen import AssistantAgent, UserProxyAgent

config_list = [
    {
        "model": "llama3.1:latest",
        "api_type": "openai",
        "base_url": "http://localhost:11434/v1",
        "api_key": "ollama"
    }
]

assistant = AssistantAgent(
    name="assistant",
    system_message=(
        "Answer questions using the supplied document. "
        "Cite page numbers when possible. "
        "If the answer is not in the document, say so."
    ),
    llm_config={
        "config_list": config_list,
        "timeout": 600,
        "cache_seed": 42
    }
)

user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    code_execution_config=False
)

print("Agents created successfully.")


Agents created successfully.


In [21]:

question = "What is the main topic of this book?"

prompt = f"""
Answer the following question using the document.

Question: {question}

Document:
{docs_content[:20000]}

Cite page numbers where possible.
Do not invent facts absent from the document.
"""

user_proxy.initiate_chat(
    assistant,
    message=prompt,
    max_turns=1
)


user_proxy (to assistant):


Answer the following question using the document.

Question: What is the main topic of this book?

Document:
[Page 2]
The 22
Immutable Laws
of Marketing
Violate Them at Y our Own Risk
Al Ries and Jack Trout
22 Laws of Marketing  10/31/02  12:23 PM  Page 2

[Page 3]
Dedicated to the elimination of
myths and misconceptions
from the marketing process
22 Laws of Marketing  10/31/02  12:23 PM  Page 3

[Page 4]
Contents
Introduction ix
1. The Law of Leadership 
2. The Law of the Category 
3. The Law of the Mind 
4. The Law of Perception 
5. The Law of Focus 
6. The Law of Exclusivity 
7. The Law of the Ladder 
8. The Law of Duality 
9. The Law of the Opposite 
10. The Law of Division 
11. The Law of Perspective 
12. The Law of Line Extension 
13. The Law of Sacrifice 
14. The Law of Attributes 
15. The Law of Candor 
16. The Law of Singularity 
17. The Law of Unpredictability 
18. The Law of Success 
19. The Law of Failure 
20.The Law of Hype 
21. The Law of Acce

ChatResult(chat_id=None, chat_history=[{'content': '\nAnswer the following question using the document.\n\nQuestion: What is the main topic of this book?\n\nDocument:\n[Page 2]\nThe 22\nImmutable Laws\nof Marketing\nViolate Them at Y our Own Risk\nAl Ries and Jack Trout\n22 Laws of Marketing  10/31/02  12:23 PM  Page 2\n\n[Page 3]\nDedicated to the elimination of\nmyths and misconceptions\nfrom the marketing process\n22 Laws of Marketing  10/31/02  12:23 PM  Page 3\n\n[Page 4]\nContents\nIntroduction ix\n1. The Law of Leadership \n2. The Law of the Category \n3. The Law of the Mind \n4. The Law of Perception \n5. The Law of Focus \n6. The Law of Exclusivity \n7. The Law of the Ladder \n8. The Law of Duality \n9. The Law of the Opposite \n10. The Law of Division \n11. The Law of Perspective \n12. The Law of Line Extension \n13. The Law of Sacrifice \n14. The Law of Attributes \n15. The Law of Candor \n16. The Law of Singularity \n17. The Law of Unpredictability \n18. The Law of Success 

# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [23]:

question = "What is the additional topic of this book?"

prompt = f"""
Answer the following question using the document.

Question: {question}

Document:
{docs_content[:20000]}

Cite page numbers where possible.
Do not invent facts absent from the document.
"""

user_proxy.initiate_chat(
    assistant,
    message=prompt,
    max_turns=1
)


user_proxy (to assistant):


Answer the following question using the document.

Question: What is the additional topic of this book?

Document:
[Page 2]
The 22
Immutable Laws
of Marketing
Violate Them at Y our Own Risk
Al Ries and Jack Trout
22 Laws of Marketing  10/31/02  12:23 PM  Page 2

[Page 3]
Dedicated to the elimination of
myths and misconceptions
from the marketing process
22 Laws of Marketing  10/31/02  12:23 PM  Page 3

[Page 4]
Contents
Introduction ix
1. The Law of Leadership 
2. The Law of the Category 
3. The Law of the Mind 
4. The Law of Perception 
5. The Law of Focus 
6. The Law of Exclusivity 
7. The Law of the Ladder 
8. The Law of Duality 
9. The Law of the Opposite 
10. The Law of Division 
11. The Law of Perspective 
12. The Law of Line Extension 
13. The Law of Sacrifice 
14. The Law of Attributes 
15. The Law of Candor 
16. The Law of Singularity 
17. The Law of Unpredictability 
18. The Law of Success 
19. The Law of Failure 
20.The Law of Hype 
21. The Law o

ChatResult(chat_id=None, chat_history=[{'content': '\nAnswer the following question using the document.\n\nQuestion: What is the additional topic of this book?\n\nDocument:\n[Page 2]\nThe 22\nImmutable Laws\nof Marketing\nViolate Them at Y our Own Risk\nAl Ries and Jack Trout\n22 Laws of Marketing  10/31/02  12:23 PM  Page 2\n\n[Page 3]\nDedicated to the elimination of\nmyths and misconceptions\nfrom the marketing process\n22 Laws of Marketing  10/31/02  12:23 PM  Page 3\n\n[Page 4]\nContents\nIntroduction ix\n1. The Law of Leadership \n2. The Law of the Category \n3. The Law of the Mind \n4. The Law of Perception \n5. The Law of Focus \n6. The Law of Exclusivity \n7. The Law of the Ladder \n8. The Law of Duality \n9. The Law of the Opposite \n10. The Law of Division \n11. The Law of Perspective \n12. The Law of Line Extension \n13. The Law of Sacrifice \n14. The Law of Attributes \n15. The Law of Candor \n16. The Law of Singularity \n17. The Law of Unpredictability \n18. The Law of Su